# Test IRIS Telescope Integration (Mixed Protocols)

Ten notatnik służy do interaktywnego testowania nowej implementacji `TreeIrisObservatory` oraz konektorów `Pilar` i `IrisCCD`.

**Co robi ten notatnik?**
1. Uruchamia w tle lokalne serwery Mock (TCP dla Pilar, UDP dla IrisCCD).
2. Podmienia konfigurację, aby łączyć się z `localhost` zamiast prawdziwym sprzętem.
3. Inicjalizuje drzewo urządzeń IRIS.
4. Pozwala wykonywać komendy `get` i `put` bezpośrednio z poziomu komórek.

In [ ]:
import asyncio
import logging
import sys
import os
from unittest.mock import MagicMock

# Dodajemy ścieżkę do głównego katalogu projektu, aby widzieć moduły obsrv
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../')))

from obsrv.tree_components.specialized_components.tree_iris import TreeIrisObservatory
from obsrv.ob_config import SingletonConfig

# Konfiguracja logowania (aby widzieć co się dzieje w konektorach)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger("JupyterIris")

## 1. Definicja Mocków Sprzętowych
Poniższe klasy symulują działanie fizycznego sprzętu (Teleskopu Pilar i Kamery UDP).

In [ ]:
class MockPilarServer:
    """Prosty serwer TCP udający teleskop Pilar."""
    def __init__(self, port=5950):
        self.server = None
        self.host = '127.0.0.1'
        self.port = port
        self.responses = {
            "GET OBJECT.EQUATORIAL.RA": ("OBJECT.EQUATORIAL.RA=10.5", True),
            "GET OBJECT.EQUATORIAL.DEC": ("OBJECT.EQUATORIAL.DEC=-20.0", True),
            "SET POINTING.SETUP.FOCUS.POSITION=2000": ("POINTING.SETUP.FOCUS.POSITION=2000", True),
        }

    async def handle_client(self, reader, writer):
        try:
            while True:
                data = await reader.readline()
                if not data: break
                line = data.decode().strip()
                if not line: continue
                
                parts = line.split(' ', 1)
                if len(parts) < 2: continue
                cmd_id, command = parts[0], parts[1]
                
                # Logika odpowiedzi
                if command in self.responses:
                    val, ok = self.responses[command]
                    writer.write(f"{cmd_id} {val}\n".encode())
                    status = "COMMAND COMPLETE" if ok else "COMMAND FAILED"
                    writer.write(f"{cmd_id} {status}\n".encode())
                else:
                    writer.write(f"{cmd_id} COMMAND COMPLETE\n".encode())
                await writer.drain()
        except Exception:
            pass

    async def start(self):
        self.server = await asyncio.start_server(self.handle_client, self.host, self.port)
        print(f"[MockPilar] Listening on TCP {self.host}:{self.port}")
        asyncio.create_task(self.server.serve_forever())

class MockIrisCcdProtocol(asyncio.DatagramProtocol):
    def datagram_received(self, data, addr):
        msg = data.decode().strip()
        resp = "**** OKAY 0" if msg == "sync" else "**** OKAY 1"
        if "temp" in msg: resp = "**** OKAY -15.5"
        self.transport.sendto(resp.encode(), addr)
        
class MockIrisCcdServer:
    """Prosty serwer UDP udający kamerę."""
    async def start(self, port=8888):
        loop = asyncio.get_running_loop()
        self.transport, _ = await loop.create_datagram_endpoint(
            lambda: MockIrisCcdProtocol(), local_addr=('127.0.0.1', port)
        )
        self.transport.protocol.transport = self.transport
        print(f"[MockIris] Listening on UDP 127.0.0.1:{port}")

## 2. Uruchomienie Mocków i Konfiguracja
Uruchamiamy serwery w tle i podmieniamy `SingletonConfig`, aby IRIS wskazywał na `localhost`.

In [ ]:
# Start serwerów
pilar_mock = MockPilarServer()
iris_mock = MockIrisCcdServer()
await pilar_mock.start()
await iris_mock.start()

# Konfiguracja testowa
test_config = {
    'iris': {
        'observatory': {
            'protocol': 'ignored',
            'components': {
                'mount': {'kind': 'telescope', 'device_number': 0, 'protocol': 'pilar', 'address': '127.0.0.1:5950', 'type': 'az'},
                'focuser': {'kind': 'focuser', 'device_number': 0, 'protocol': 'pilar', 'address': '127.0.0.1:5950'},
                'camera': {'kind': 'camera', 'device_number': 0, 'protocol': 'iris_ccd', 'address': '127.0.0.1:8888'},
                'dome': {'kind': 'dome', 'device_number': 0, 'protocol': 'alpaca', 'address': 'http://127.0.0.1:11111/api/v1'}
            }
        }
    }
}

# Patch configu
SingletonConfig.get_config = MagicMock(return_value=test_config)

## 3. Inicjalizacja Drzewa IRIS
Tworzymy obiekt `TreeIrisObservatory`. W tym momencie konektory są przygotowywane (lazy loading).

In [ ]:
tree_iris = TreeIrisObservatory('iris-notebook', observatory_name='iris')
print("Drzewo IRIS zainicjalizowane. Dostępne komponenty:")
for child in tree_iris._observatory.children:
    print(f" - {child}")

## 4. Testy Protokołu Pilar (Teleskop)
Pobieramy `rightascension` oraz testujemy równoległość zapytań.

In [ ]:
# Proste pobranie
ra = await tree_iris._observatory.mount.get('rightascension')
print(f"Right Ascension: {ra} stopni (Mock zwraca 10.5h)")

In [ ]:
# Test równoległości - wysyłamy 5 zapytań na raz
tasks = [
    tree_iris._observatory.mount.get('rightascension'),
    tree_iris._observatory.mount.get('declination'),
    tree_iris._observatory.mount.get('rightascension'),
    tree_iris._observatory.mount.get('declination'),
    tree_iris._observatory.mount.get('rightascension')
]
results = await asyncio.gather(*tasks)
print(f"Wyniki zapytań równoległych: {results}")

In [ ]:
# Test PUT (ruch focusera)
resp = await tree_iris._observatory.focuser.put('move', Position=2000)
print(f"Odpowiedź Focusera: {resp}")

## 5. Testy Protokołu IrisCCD (Kamera)
Testujemy komunikację po UDP.

In [ ]:
# Pobranie statusu kamery
state = await tree_iris._observatory.camera.get('camerastate')
print(f"Camera State (UDP): {state}")

# Pobranie temperatury (symulowane)
temp = await tree_iris._observatory.camera.get('ccdtemperature')
print(f"Camera Temp: {temp}")

## 6. Czyszczenie
Zatrzymanie serwerów po testach.

In [ ]:
pilar_mock.server.close()
await pilar_mock.server.wait_closed()
iris_mock.transport.close()
print("Mocki zatrzymane.")